# 2. Zero-shot judges and tokenizer analysis
This runs every judge on the **test split** with the same rubric and reasoning format, then writes the tables into `README.md`.

Judges:
- the string-matching shortcut;
- three small open models in 4-bit (Africa-focused Tiny Aya Earth, multilingual Qwen3-4B, English-centric Llama 3.2 3B);
- Gemini.

**Runtime:** T4 GPU. **Needs:** `GEMINI_API_KEY` and `HF_TOKEN` in Secrets, and access granted to the gated models.

In [ ]:
REPO_URL = "https://github.com/KingQuinn/cross-lingual-groundedness-judge.git"
import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!git pull -q
!pip -q install -r requirements.txt

In [ ]:
# Everything that costs API calls or GPU time is saved to Google Drive, so a
# disconnect never loses work; re-running a cell resumes where it stopped.
from google.colab import drive, userdata
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cross-lingual-judge"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# Tiny Aya and Llama are gated: accept each licence on its Hugging Face page
# (logged in), then add a Hugging Face token as the Colab secret HF_TOKEN.
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
if not os.path.exists("data/items.jsonl"):
    !mkdir -p data && cp $WORK/data/items.jsonl data/
PREDS = f"{WORK}/preds"
os.makedirs(PREDS, exist_ok=True)
LIMIT = 500   # about 500 original test items, sampled as whole documents (identical for every judge); None = all
LIM = f"--limit {LIMIT}" if LIMIT else ""
MODELS = ["CohereLabs/tiny-aya-earth", "Qwen/Qwen3-4B", "meta-llama/Llama-3.2-3B-Instruct"]

## Tokenizer fertility (tokens per word, per language)

In [ ]:
!python scripts/tokenizer_fertility.py --models {" ".join(MODELS)}
!cp results/fertility.json $WORK/fertility.json

## Shortcut baseline

In [ ]:
!python scripts/run_judge.py --judge shortcut $LIM --out_dir $PREDS

## Open models: a 3-item smoke test first, then the full run

In [ ]:
for m in MODELS:
    !python scripts/run_judge.py --judge hf:{m} --limit 1 --no_variants --out_dir /content/smoke
import glob
for f in glob.glob("/content/smoke/*.jsonl"):
    r = json.loads(open(f).readline())
    print(f"=== {f.split('/')[-1]}  gold={r['label']}  pred={r['pred']}  p_supported={r['p_supported']:.3f}  complete={r['format_complete']}")
    print(r["raw"][:600], "\n")

In [ ]:
for m in MODELS:
    !python scripts/run_judge.py --judge hf:{m} $LIM --out_dir $PREDS

## API judge (Gemini)
Note for the report: Gemini also translated the data, so it may be favoured when judging text from its own model family.

In [ ]:
API_MODEL = "gemini-3.8-flash"
!python scripts/run_judge.py --judge gemini:{API_MODEL} $LIM --out_dir $PREDS

## Analysis → README

In [ ]:
!rm -rf results/preds && mkdir -p results && cp -r $PREDS results/preds && cp $WORK/fertility.json results/
!python scripts/analyze.py
from IPython.display import Markdown, display
display(Markdown(open("results/RESULTS.md").read()))

In [ ]:
!zip -qr /content/results_to_commit.zip README.md results
from google.colab import files
files.download("/content/results_to_commit.zip")